# 07 · What held, what did not, and what to do about it

Every claim in the two proposals that could be tested against a public label matrix, with
the measurement and the verdict. Nothing here is a plan; it is what the numbers said.

The corpus throughout is RouterBench — 36,497 real prompts × 11 real commercial models,
401,467 graded cells with realised costs. Zero new inference was purchased. Held-out
evaluation is by item, never by row.

In [1]:
import sys; sys.path.insert(0, "..")
from notebooks.nbhelp import load, table, pct, show, md, finding
import numpy as np, pandas as pd

In [2]:
ov, fr, st, cs, mt, dc, gr = (load(n) for n in
    ["overview", "frontier", "staleness", "coldstart", "metric", "decomposition", "gram"])
man = load("manifest")
sw = pd.DataFrame(fr["sweep"]); s5 = sw.loc[(sw["lam_cost"]-0.05).abs().idxmin()]
sts = st["summary"]; lk = mt["result"]["oracle_luck"]
rk = mt["result"]["ranking"]["overall"]; bins = mt["result"]["bins"]["bins"]
rep = dc["replication"]

md(f"""
| # | claim | source | verdict | the measurement |
|---|---|---|---|---|
| 1 | Most queries do not need the strongest model | §3.1 | **supported** | {pct(ov['ties']['pairwise_tie_rate'])} pairwise tie rate over {ov['ties']['pairs_compared']:,} comparisons |
| 2 | Routing beats one good model on cost at near-equal quality | AC-1 | **supported** | {pct(s5['savings_vs_frontier'])} cheaper at {pct(s5['quality_vs_frontier'])} of frontier quality |
| 3 | A router trained once measurably decays | §14.1 | **supported** | {sts['frozen_decay']:.3f} decay over {len(st['result']['weeks'])} weeks; crosses below best-single at week {sts['week_crossed_below_best_single']} |
| 4 | The decay is about evaluation freshness | §1 framing | **not supported** | {sts['attribution_new_models']/(sts['attribution_new_models']+sts['attribution_fresher_data']):.0%} of it is new-model access; refitting recovers {sts['attribution_fresher_data']:+.3f} |
| 5 | A useful router needs under 1 MB and no GPU | O3 | **supported** | {sw.iloc[0]['artifact_kb']:.0f} KB, CPU only, O(d²) updates |
| 6 | Uninformative batches make the score noise | §6 | **supported** | SD {bins[0]['raw_sd_mean']:.4f} vs {bins[-1]['raw_sd_mean']:.4f}, low vs high information |
| 6b | ...because similar models converge oracle and baseline | §6 | **not supported** | information correlates {mt['result']['diagnosis']['corr_info_vs_model_spread']:+.2f} with model spread — the causal story is inverted |
| 7 | Information-aware shrinkage fixes it | §6.1 | **supported** | ranking concordance {rk['raw_concordance']:.3f} → {rk['shrunk_concordance']:.3f} |
| 8 | Normalised regret below 0.6 is achievable | AC-1 | **not supported** | {pct(lk['mean_luck_share_of_gap'])} of the oracle gap is unattainable luck; a clairvoyant per-task oracle scores 0.078 |
| 9 | Low-rank item factors bridge to a feature-space prior | §5.1 | **not supported** | bridge R² = {cs['bridge']['bridge_r2']:.3f}; the proposal's own gate fails |
| 10 | Probe count tracks how well the pool explains a new model | §5.3 | **mixed** | τ² vs probe count r = {cs['lomo']['summary']['tau2_vs_probe_correlation']['derived (§5.2)']:+.2f}, but n = 1 material model |
| 11 | Component-wise γ_q/γ_t beats one shared γ | §4 | **{'mixed' if rep['high_drift']['decomposition']['supported'] != rep['default']['decomposition']['supported'] else ('supported' if rep['high_drift']['decomposition']['supported'] else 'not supported')}** | {rep['high_drift']['decomposition']['mean_regret_reduction']:+.4f} ± {rep['high_drift']['decomposition']['std_error']:.4f} under engineered drift; {rep['default']['decomposition']['mean_regret_reduction']:+.4f} ± {rep['default']['decomposition']['std_error']:.4f} otherwise |
| 12 | Price should be read, not fitted | §8.7 | **supported** | live-read absorbs a quality shock {rep['default']['transient_by_shock_kind']['quality']['live_read_advantage']:+.4f} ± {rep['default']['transient_by_shock_kind']['quality']['live_read_se']:.4f} better; and FR-16 is unsatisfiable by a fitted-price router at any γ |
| 13 | γ ≈ 0.999 is a sensible default | §8.4 | **not supported** | γ = 1.0 beats it by {(next(r for r in dc['gamma_tuning']['shared'] if abs(r['gamma']-0.999)<1e-9)['mean_regret'] - dc['gamma_tuning']['best_shared']['mean_regret']):+.4f} regret |
| 14 | One shared Gram matrix serves every model | §8.3 | **not supported** | exact at uniform coverage; under-predicts by (1−coverage) otherwise, costing {gr['summary']['worst_utility_gap']:.4f} utility |
""")


| # | claim | source | verdict | the measurement |
|---|---|---|---|---|
| 1 | Most queries do not need the strongest model | §3.1 | **supported** | 57.8% pairwise tie rate over 2,007,335 comparisons |
| 2 | Routing beats one good model on cost at near-equal quality | AC-1 | **supported** | 44.5% cheaper at 99.4% of frontier quality |
| 3 | A router trained once measurably decays | §14.1 | **supported** | 1.022 decay over 26 weeks; crosses below best-single at week 5 |
| 4 | The decay is about evaluation freshness | §1 framing | **not supported** | 99% of it is new-model access; refitting recovers +0.010 |
| 5 | A useful router needs under 1 MB and no GPU | O3 | **supported** | 358 KB, CPU only, O(d²) updates |
| 6 | Uninformative batches make the score noise | §6 | **supported** | SD 0.1628 vs 0.1113, low vs high information |
| 6b | ...because similar models converge oracle and baseline | §6 | **not supported** | information correlates -0.55 with model spread — the causal story is inverted |
| 7 | Information-aware shrinkage fixes it | §6.1 | **supported** | ranking concordance 0.805 → 0.896 |
| 8 | Normalised regret below 0.6 is achievable | AC-1 | **not supported** | 50.4% of the oracle gap is unattainable luck; a clairvoyant per-task oracle scores 0.078 |
| 9 | Low-rank item factors bridge to a feature-space prior | §5.1 | **not supported** | bridge R² = 0.272; the proposal's own gate fails |
| 10 | Probe count tracks how well the pool explains a new model | §5.3 | **mixed** | τ² vs probe count r = +0.47, but n = 1 material model |
| 11 | Component-wise γ_q/γ_t beats one shared γ | §4 | **not supported** | +0.0021 ± 0.0010 under engineered drift; -0.0005 ± 0.0006 otherwise |
| 12 | Price should be read, not fitted | §8.7 | **supported** | live-read absorbs a quality shock +0.0080 ± 0.0033 better; and FR-16 is unsatisfiable by a fitted-price router at any γ |
| 13 | γ ≈ 0.999 is a sensible default | §8.4 | **not supported** | γ = 1.0 beats it by +0.0652 regret |
| 14 | One shared Gram matrix serves every model | §8.3 | **not supported** | exact at uniform coverage; under-predicts by (1−coverage) otherwise, costing 0.0558 utility |


## The four findings that change what should be built

**1. Onboarding beats re-grading.** The decay is real, and it is almost entirely about
being able to select models that did not exist at training time — not about the freshness
of the data on the models you already have. That inverts the cost priority in §18.2: the
daily full-pool sweep over fresh items is buying the less valuable half. A catalogue
watcher plus a fast, reliable cold-start probe is buying the more valuable half, and it
is far cheaper.

**2. Per-model Gram matrices, not one shared.** Worth more than every other estimator
change measured here, combined. The shared-matrix shortcut is exact under uniform
coverage and silently stops selecting a model whose coverage is thin — which is precisely
the new-model case that finding 1 says matters most, and precisely the sampling-plan case
§18.2 recommends. The fix costs K solves of a 64×64 system and one extra matrix per
model.

**3. The scoring rule needs a reachable ceiling.** Half of §8.8's denominator is luck no
policy can capture. That deflates every score, makes AC-1's threshold unreachable by
construction, and — because §16.1 pays emissions in proportion to these scores — pays
partly on noise. Define the oracle over grouped or conditionally-expected outcomes, and
restate AC-1 against it.

**4. Cold start fails on the model you most need it for.** Matrix-completion priors work
for a model that resembles the pool and fail for the frontier outlier: loading R² ranges
from 0.43 down to −0.008, and the −0.008 is GPT-4, whose arrival caused all the decay in
finding 1. Onboarding the models that matter needs real probing, and the probe budget
should be set by measured loading rather than by a flat ~250.

## What to drop

Contribution 1's per-component decay is detectable only under continuous drift built to
favour it, and even there the improvement is a fraction of a percent of the regret it is
improving — while doubling the Gram state the artifact has to carry. The read-versus-learn
half of it (1b) is worth keeping — for FR-16, and because a learned-cost target conflates
signals that should stay separate — but the two-γ machinery is not carrying its weight. §8.4's γ ≈ 0.999 default should be replaced by γ = 1.0 with
drift-triggered forgetting, since always-on forgetting measurably costs regret on a pool
that is mostly stationary.

Contribution 2's bridge fails its own precondition. The *idea* behind §5.3 — that probe
budget should be a function of measured loading rather than a constant — survives and is
worth keeping; the specific item-space-to-feature-space derivation does not.

## Limits of this evidence

Stated plainly, because several of them bound the conclusions above.

- **One corpus, eleven models, 2023 vintage.** RouterBench's pool ends in December 2023.
  The staleness result is one trajectory through one pool's history, not a distribution
  over them.
- **Binary grading.** Most items are multiple-choice, so per-item outcomes carry a lot of
  luck. That is the direct cause of finding 3 and it also caps how much per-item signal
  any router can extract.
- **No item dates.** RouterBench does not date its prompts, so the replay holds the item
  distribution fixed. Workload drift and contamination — two of the five causes §3.3
  lists — are not measured here at all.
- **Synthetic shocks.** Every drift result in notebook 06 rests on injected changes,
  because a static snapshot contains no real drift. The magnitudes are plausible; they are
  not observed.
- **n = 1 on the case that matters.** Only one of eleven models had a material onboarding
  gap, so findings 4 and 10 rest on a single instance.
- **Feature map is deliberately cheap.** Hashed n-grams plus surface features, not a
  sentence encoder. §3.1 says encoder scale barely moves routing accuracy; that was taken
  on trust here rather than verified, so the absolute scores are a floor.

In [3]:
md(f"""
### Reproducing this

```
python scripts/fetch_data.py            # ~95 MB, once
python scripts/run_all.py               # ~{man['elapsed_seconds']/60:.0f} min, CPU only
python scripts/execute_notebooks.py     # re-renders these notebooks
```

Seed {man['seed']}, d = {man['feature_dim']}, {man['n_train']:,} train / {man['n_test']:,} test
items disjoint by item. Every experiment reads files on disk and touches no network, so
two runs produce the same numbers.
""")


### Reproducing this

```
python scripts/fetch_data.py            # ~95 MB, once
python scripts/run_all.py               # ~2 min, CPU only
python scripts/execute_notebooks.py     # re-renders these notebooks
```

Seed 0, d = 64, 25,000 train / 11,497 test
items disjoint by item. Every experiment reads files on disk and touches no network, so
two runs produce the same numbers.
